# Student Dropout Prediction - Stage 2


## Colab / Local Setup

Local execution still works from the cloned repo in VS Code / WSL.

In Google Colab, start each new runtime by cloning the repo into `/content/student-dropout-prediction`, installing dependencies, and then running the setup cell below:

```bash
!git clone <repo-url> /content/student-dropout-prediction
%cd /content/student-dropout-prediction
!pip install -r requirements.txt
```

If you clone into a different folder name, set `COLAB_PROJECT_REPO` before running the setup cell.


In [ ]:
import os
import sys
from pathlib import Path


def _is_colab_runtime() -> bool:
    return 'google.colab' in sys.modules


def _resolve_repo_root() -> Path:
    repo_name = os.environ.get('COLAB_PROJECT_REPO', 'student-dropout-prediction')
    search_roots = [Path.cwd().resolve()]

    if _is_colab_runtime():
        colab_repo_root = Path('/content') / repo_name
        search_roots.append(colab_repo_root)

    seen = set()
    for root in search_roots:
        for candidate in [root, *root.parents]:
            if candidate in seen:
                continue
            seen.add(candidate)
            if all((candidate / part).exists() for part in ('src', 'data', 'notebooks')):
                return candidate

    raise FileNotFoundError(
        'Could not locate the repository root. In Colab, clone the repo into '
        f'/content/{repo_name} or set COLAB_PROJECT_REPO to the cloned folder name.'
    )


IS_COLAB = _is_colab_runtime()
BOOTSTRAP_PROJECT_ROOT = _resolve_repo_root()
BOOTSTRAP_NOTEBOOK_DIR = BOOTSTRAP_PROJECT_ROOT / 'notebooks'

if IS_COLAB and Path.cwd().resolve() != BOOTSTRAP_NOTEBOOK_DIR:
    os.chdir(BOOTSTRAP_NOTEBOOK_DIR)

if str(BOOTSTRAP_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_PROJECT_ROOT))

print(f'Running in Colab: {IS_COLAB}')
print(f'Project root: {BOOTSTRAP_PROJECT_ROOT}')
print(f'Working directory: {Path.cwd().resolve()}')


In [ ]:
import json
import os
import random
import ssl
import sys
from pathlib import Path

import joblib
import keras_tuner as kt
import matplotlib.pyplot as plt
import numpy as np
import optuna
import optuna.visualization as vis
import pandas as pd
import requests
import seaborn as sns
import shap
import tensorflow as tf
import xgboost as xgb
from IPython.display import HTML, display
from sklearn.metrics import mean_squared_error, r2_score
from tensorflow.keras import mixed_precision
from tensorflow.keras.callbacks import EarlyStopping as KerasEarlyStopping
from xgboost.callback import EarlyStopping as XgbEarlyStopping

from src.config import (
    DEFAULT_RANDOM_SEED,
    MANUAL_REVIEW_DEFAULT,
    RUNTIME_DEFAULTS,
    STABILITY_SEEDS,
    TARGET_COLUMN,
    TUNING_ARTIFACT_FILENAMES,
)
from src.notebook_helpers import (
    analyze_hp_importance,
    build_binary_classifier,
    build_models_to_plot,
    choose_n_jobs,
    compute_confusion_matrix_elements,
    compute_performance_metrics,
    create_train_val_test_split_and_scale,
    evaluate_and_store_model,
    plot_confusion_matrix,
    plot_model_confusion_matrix,
    plot_grouped_feature_importance,
    plot_roc_and_pr_curves,
    print_model_metrics,
    set_seed,
)
from src.paths import (
    DATA_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    PROJECT_ROOT,
    TUNING_DIR,
    ensure_directories,
    get_stage_data_path,
    get_stage_paths,
)
from src.tuning_utils import (
    best_params_record,
    load_model_weights,
    load_saved_best_params,
    run_keras_tuner as shared_run_keras_tuner,
    run_optuna_xgb as shared_run_optuna_xgb,
    tuner_trials_to_dataframe,
)

STAGE_NAME = 'stage_2'
PREVIOUS_STAGE = 'stage_1'
stage_paths = get_stage_paths(STAGE_NAME, previous_stage=PREVIOUS_STAGE)
STAGE_DATA_PATH = get_stage_data_path(STAGE_NAME)
DATA_CACHE_DIR = stage_paths['DATA_CACHE_DIR']
STAGE_TUNING_DIR = stage_paths['STAGE_TUNING_DIR']
XGB_TUNING_DIR = stage_paths['XGB_TUNING_DIR']
NN_TUNING_DIR = stage_paths['NN_TUNING_DIR']
PREV_XGB_TUNING_DIR = stage_paths['PREV_XGB_TUNING_DIR']
PREV_NN_TUNING_DIR = stage_paths['PREV_NN_TUNING_DIR']
STAGE_MODEL_DIR = stage_paths['STAGE_MODEL_DIR']
XGB_MODEL_DIR = stage_paths['XGB_MODEL_DIR']
NN_MODEL_DIR = stage_paths['NN_MODEL_DIR']

ensure_directories(
    DATA_CACHE_DIR,
    TUNING_DIR,
    STAGE_TUNING_DIR,
    XGB_TUNING_DIR,
    NN_TUNING_DIR,
    MODELS_DIR,
    STAGE_MODEL_DIR,
    XGB_MODEL_DIR,
    NN_MODEL_DIR,
    FIGURES_DIR,
)

LOAD_SAVED_TUNING = True
RUN_XGB_TUNING = False
RUN_NN_TUNING = False
RESUME_XGB_TUNING = True
RESUME_NN_TUNING = True

XGB_N_TRIALS = 2000
NN_N_TRIALS = 200

RELOAD_DATA_CACHE = RUNTIME_DEFAULTS['RELOAD_DATA_CACHE']
MANUAL_REVIEW = MANUAL_REVIEW_DEFAULT
SEED = DEFAULT_RANDOM_SEED


def run_keras_tuner(
    max_trials,
    project_name,
    X_train,
    y_train,
    X_val,
    y_val,
    executions_per_trial=1,
    overwrite=False,
    hp_bs=64,
    artifact_dir=None,
    run_search=True,
    load_saved=True,
    resume_search=True,
    artifact_prefix='',
):
    def build_model_for_tuner(hp):
        return build_binary_classifier(
            input_dim=X_train.shape[1],
            units=hp.Int('units', **SEARCH_SPACE['units']),
            layers=hp.Int('layers', **SEARCH_SPACE['layers']),
            activation=hp.Choice('activation', SEARCH_SPACE['activation']),
            optimizer=hp.Choice('optimizer', SEARCH_SPACE['optimizer']),
            lr=hp.Choice('lr', SEARCH_SPACE['lr']),
            dropout=hp.Float('dropout', **SEARCH_SPACE['dropout']),
            l2_strength=hp.Choice('l2_strength', SEARCH_SPACE['l2_strength']),
        )

    early_stop = KerasEarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        min_delta=1e-4,
    )

    return shared_run_keras_tuner(
        kt_module=kt,
        build_model=build_model_for_tuner,
        max_trials=max_trials,
        project_name=project_name,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        early_stopping_callback=early_stop,
        seed=SEED,
        artifact_dir=artifact_dir or NN_TUNING_DIR,
        executions_per_trial=executions_per_trial,
        overwrite=overwrite,
        hp_bs=hp_bs,
        run_search=run_search,
        load_saved=load_saved,
        resume_search=resume_search,
        artifact_prefix=artifact_prefix,
    )


def run_optuna_xgb(
    n_trials,
    study_name,
    X_train,
    y_train,
    X_val,
    y_val,
    seed=42,
    overwrite=False,
    n_estimators=3000,
    early_stopping_rounds=30,
    n_jobs=-1,
    artifact_dir=None,
    run_search=True,
    load_saved=True,
    resume_search=True,
):
    return shared_run_optuna_xgb(
        study_name=study_name,
        search_space=XGB_SEARCH_SPACE,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        artifact_dir=artifact_dir or XGB_TUNING_DIR,
        model_dir=XGB_MODEL_DIR,
        seed=seed,
        overwrite=overwrite,
        n_trials=n_trials,
        n_estimators=n_estimators,
        early_stopping_rounds=early_stopping_rounds,
        n_jobs=n_jobs,
        run_search=run_search,
        load_saved=load_saved,
        resume_search=resume_search,
        early_stopping_callback_factory=lambda rounds: XgbEarlyStopping(
            rounds=rounds,
            metric_name='auc',
            maximize=True,
        ),
    )

CACHE_PATH = DATA_CACHE_DIR / 'stage_2_data.h5'
hdf_key = 'stage_2_data'

if not RELOAD_DATA_CACHE and CACHE_PATH.exists():
    try:
        stage_2_data = pd.read_hdf(CACHE_PATH, hdf_key)
        print(f'Loaded cached Stage 2 data from {CACHE_PATH}')
    except (OSError, ValueError, KeyError) as e:
        print(f'Failed to load cached Stage 2 data, reloading from CSV: {e}')
        stage_2_data = pd.read_csv(STAGE_DATA_PATH)
        stage_2_data.to_hdf(CACHE_PATH, key=hdf_key, mode='w')
        print(f'Rebuilt cache at {CACHE_PATH}')
else:
    stage_2_data = pd.read_csv(STAGE_DATA_PATH)
    stage_2_data.to_hdf(CACHE_PATH, key=hdf_key, mode='w')
    print(f'Loaded Stage 2 public dataset and refreshed cache at {CACHE_PATH}')

stage_2_data_unmodified = stage_2_data.copy()


**Stage 2: Pre-processing instructions**

- Remove any columns not useful in the analysis (LearnerCode).
- Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set).
- Remove columns with >50% data missing.
- Perform ordinal encoding for ordinal data.
- Perform one-hot encoding for all other categorical data.
- Choose how to engage with missing values, which can be done in one of two ways for this project:
  *   Impute the rows with appropriate values.
  *   Remove rows with missing values but ONLY in cases where rows with missing values are minimal: <2% of the overall data.



In [ ]:
stage_2_data.head()


## Data Inspection

First we'll review the data and perform any data cleanse or pre-processing.

### Comparison with Stage 1 Data

Stage 2 data is known to be the original Stage 1 dataset with the addition of attendance metrics, including authorised absence and unauthorised absence counts.

EDA and feature engineering have already been completed on stage_1_data, so we will verify the following:

- Columns that do not match those in the Stage 1 dataset
- Any differences in index-aligned rows for the columns common to both datasets

If there are no differences between the original Stage 1 columns and the corresponding columns in Stage 2, then EDA can be focused on the newly added columns only.

In [ ]:
# determine the number of rows that differ between the two datasets for the 
# common columns
common_cols = stage_1_data_unmodified.columns.intersection(
    stage_2_data_unmodified.columns
)

# select only the common columns from both datasets to compare the rows and
# identify any differences in the data for those columns
a = stage_1_data_unmodified[common_cols]
b = stage_2_data_unmodified[common_cols]

# create a boolean mask to identify rows that differ between the two datasets
# for the common columns, accounting for potential NaN values by treating rows
# as the same if they are both NaN in a column
diff_mask = ~((a == b) | (a.isna() & b.isna())).all(axis=1)

# sum the boolean mask to get the total count of rows that differ between the 
# two datasets for the common columns
diff_count = diff_mask.sum()

print("Rows that differ:", diff_count)

new_cols_stage2 = list(stage_2_data_unmodified.columns.difference(common_cols))

print("New Columns in Stage 2:\n", new_cols_stage2)

# copy the new columns to a new dataframe for easier analysis and to understand
# the additional information available in the stage 2 dataset that was not present
# in stage 1, which may be useful for further analysis or modelling.
stage_2_new_cols = stage_2_data_unmodified[new_cols_stage2]

# View the metadata with the info function.
print("Datatypes: \n" + str(stage_2_new_cols.info()) + "\n")


There are two new columns: 

- AuthorisedAbsenceCount (float64)
- UnauthorisedAbsenceCount (float64)

All other columns match stage 1 data exactly, therefore we will focus EDA on the two new columns only and then add them to the stage 1 data post encoding if they are kept.

### Data Quality Checks

We will perform basic data quality checks, view the dataframe metadata to determine assigned datatypes, and determine the size of the dataset.

Data quality checks (numeric fields):

- Missing data checks
- Count of unique values
- Count of null rows


In [ ]:
# Check for null values
print(
    "\nData quality check (1) detect null values in columns:\n"
    + str(
        stage_2_new_cols.isna()
        .sum()
        .to_frame("null_count")
        .assign(null_pct=lambda x: (x / len(stage_1_data) * 100).round(2))
        .sort_values(by="null_count", ascending=False)
    )
    + "\n"
)


# count unique values in each column and display count & percentage
print(
    "\nData quality check (2) unique values in each column:\n"
    + str(
        stage_2_new_cols.nunique()
        .to_frame("distinct_count")
        .assign(distinct_pct=lambda x: (x / len(stage_1_data) * 100).round(2))
        .sort_values(by="distinct_count", ascending=True)
    )
    + "\n"
)

# count the number of rows where the both new columns are null to understand the
# extent of missing data in the new columns and to assess whether it may be a
# significant issue for analysis or modeling.
missing_both = stage_2_new_cols.isna().all(axis=1).sum()
print(
    f"Number of rows where both new columns are null: {missing_both} "
    f"({(missing_both / len(stage_1_data) * 100):.2f}%)"
)


**Initial data quality checks interpretation**

---

***Data Summary***

- 25059 records
- All columns match the Stage 1 dataset, with the addition of two new columns:
    - AuthorisedAbsenceCount
    - UnauthorisedAbsenceCount                            


***Null values in the fields:***

New fields have 208 (0.83%) null values:

- AuthorisedAbsenceCount
- UnauthorisedAbsenceCount   

These are on the same row, so the values are highly likely represent missing data for the records in question 

***Missing Data***

As these rows represent only 0.83% of the dataset, we could either remove them or impute values.

We may also create a new feature missing_absence_data to flag rows where this information is not available.

We will examine the feature distributions and the relationship between missingness and the target variable before deciding on the appropriate course of action.

--

### Missing Data Analysis

In [ ]:
##investigate null and zero values in the numeric columns to see if there are any
# patterns in the dropout rate for those with null or zero values compared to
# those with positive values.

stage_2_new_cols = stage_2_new_cols.copy()
stage_2_new_cols["dropout"] = stage_1_data["dropout"]

for col in ["AuthorisedAbsenceCount", "UnauthorisedAbsenceCount"]:
    summary = (
        stage_2_new_cols.assign(
            category=lambda df: np.select(
                [df[col].isna(), df[col] == 0],
                ["NULL", "ZERO"],
                default="PRESENT",
            )
        )
        .groupby("category")["dropout"]
        .agg(count="size", dropout_rate="mean")
        .reset_index()
    )

    plt.figure(figsize=(6, 4))

    ax = sns.barplot(
        data=summary,
        x="category",
        y="dropout_rate",
        order=["NULL", "ZERO", "PRESENT"],
    )

    ax.set_title(f"Dropout rate by {col}")
    ax.set_ylim(0, 1)

    for i, row in summary.iterrows():
        ax.text(
            i, row["dropout_rate"] + 0.02, f"n={row['count']}", ha="center"
        )

    plt.tight_layout()
    plt.show()



### Exploratory Data Analysis

Generate descriptive statistics and visualise the data to explore patterns, distributions, and trends in the data.

### Descriptive Statistics

We will now generate descriptive statistics (including percentile values for all numeric columns in the dataset), and document any insights this provides about the data.

In [ ]:
# Use df.describe to generate descriptive stats of new columns to understand the
#  distribution of values, central tendency, and variability in the new columns,
#  which can inform further analysis or modelling decisions.

stage_2_new_cols.describe(
    include="all", percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
).fillna(0).T.sort_index()


**Visualise the data**

Visualise the numeric data with box-plots and histograms to further understand the data patterns, distributions, and trends.



In [ ]:
# choose columns to plot - exclude categorical columns
cols = [
    "UnauthorisedAbsenceCount",
    "AuthorisedAbsenceCount",
]

# plot histograms and boxplots for the numeric columns to visualise their 
# distributions and check for outliers

plot_distributions_and_qq(stage_2_new_cols, cols)


### EDA Review & Feature Engineering Decisions

**AuthorisedAbsenceCount**

- The distribution is highly right-skewed, with a median of 1 absence and 25% of students having no authorised absences.
- The mean (15.1) is substantially higher than the median, indicating the presence of a small number of very large values.
- The 95th percentile is 76, while the maximum reaches 292, suggesting the presence of a long tail of high-absence cases.
    - This may be indicative of disengaged or at-risk students.

**UnauthorisedAbsenceCount**

- Unauthorised absences show a much higher central tendency, with a median of 29 and a mean of 40.5.
- The distribution is again right-skewed, with the 95th percentile at 120 and a maximum of 343.
- Compared to authorised absences, unauthorised absences appear to be more common and more widely spread

The histograms and Q-Q plots confirm the right skew and presence of outliers representing a small number of students with high absence counts.

- Boxplots show many outliers, but these appear to represent genuine observations rather than data errors.
- Q-Q plots confirm that both variables deviate substantially from normality, with heavy positive skew.

Both features indicate that most students have relatively low absence counts, while a small number have very high values.

**Missing Data**

Missing absence counts show a strong association with dropout, indicating that missingness itself carries predictive signal. Records lacking absence data exhibit substantially higher dropout rates.

The rows containing missing absence values should not be removed and an imputation strategy will be implemented which:

- Ensures nulls are replaced by an appropriate value to permit model training and evaluation
- Encodes the missingness 

Absence counts exhibit right-skewed distributions, but values are not so extreme as to mandate transformation. For simplicity, no transformation will be applied, and features will be standardised prior to neural network training.

## Pre-Processing

### Imputation

To preserve the missing value signal we will add a single feature absence_values_missing. This will be set based on the AuthorisedAbsenceCount / UnauthorisedAbsenceCount values::

- Both values null: 1
- Both values not-null: 0

As both values are either present or missing simultaneously, a single absence_values_missing feature is sufficient.

- A zero placeholder value will be imputed for null values

The resultant feature interpretation is shown below:

| absence_values_missing | AuthorisedAbsenceCount | Meaning   |
| --------------- | ---------- | --------- |
| 0               | 5          | real      |
| 0               | 0          | real zero |
| 1               | 0          | missing   |


### Encoding

***Not required***: The new columns are either already binary encoded (absence_values_missing), or numeric count values; therefore no additional encoding is required.

As we are adding these features to the pre-processed Stage 1 data, the new features can be appended to the Stage 1 data frame saved earlier, after encoding but before scaling.


### Scaling

We could consider using RobustScaler as it is more robust to outliers, however to preserve consistency and comparability between models in the previous stage we will continue to use StandardScaler.

The combined post-encoding Stage 1 data and the new Stage 2 features must be scaled to account for the additional AuthorisedAbsenceCount / UnauthorisedAbsenceCount features prior to neural network training and evaluation.


In [ ]:

# create new absence_values_missing column to indicate rows where both absence 
# count columns are null
stage_2_new_cols = stage_2_new_cols.assign(
    absence_values_missing=lambda df: (
        df[["UnauthorisedAbsenceCount", "AuthorisedAbsenceCount"]]
        .isna()
        .all(axis=1)
        .astype(int)
    )
)

# fill null values in the absence count columns with 0 placeholder
stage_2_new_cols["UnauthorisedAbsenceCount"] = stage_2_new_cols[
    "UnauthorisedAbsenceCount"
].fillna(0)
stage_2_new_cols["AuthorisedAbsenceCount"] = stage_2_new_cols[
    "AuthorisedAbsenceCount"
].fillna(0)

# add the new columns back to the post-encoding stage 1 dataset
stage_2_data_encoded = pd.concat(
    [
        stage_1_data_encoded,
        stage_2_new_cols[
            [
                "UnauthorisedAbsenceCount",
                "AuthorisedAbsenceCount",
                "absence_values_missing",
            ]
        ],
    ],
    axis=1,
)

print("Stage 2 dataset shape:", stage_2_data_encoded.shape)


## Train / validation / test split

We will create a train, test, and validation data set. Training set split 80-20, with 10% of the training set used as validation.

As the target is imbalanced we will use stratified split.

We also create scaled versions of the feature matrices for neural network training, as neural networks rely on gradient-based optimisation and are sensitive to differences in feature scale

In [ ]:
stage_2_data_encoded.head()


In [ ]:
# create train, validation, and test splits for the stage 2 dataset using the 
# same function as before to ensure consistency in how we split the data for 
# modeling and evaluation.
(
    X_train,
    X_train_s,
    X_val,
    X_val_s,
    X_test,
    X_test_s,
    y_train,
    y_val,
    y_test,
    scaler,
) = create_train_val_test_split_and_scale(
    stage_2_data_encoded, stratify=True, seed=SEED
)


## Stage 2 XGBoost Model

A new XGBoost model will be trained and evaluated on the updated dataset using the best-performing hyperparameters identified during Stage 1. Model performance will then be compared with the Stage 1 results.

A new model is required because the feature set has changed, resulting in different input dimensionality.

In [ ]:

previous_stage_xgb_params = load_saved_best_params(PREV_XGB_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['best_params'])
if previous_stage_xgb_params is None:
    previous_stage_xgb_params = {
        'learning_rate': 0.14666797316704522,
        'max_depth': 8,
        'min_child_weight': 1.1079395093578468,
        'gamma': 1.8689967766791105,
        'subsample': 0.8560129261469889,
        'colsample_bytree': 0.9882795058818716,
        'reg_alpha': 0.019465311593045143,
        'reg_lambda': 0.03081909706850276,
    }
    print(
        'No saved Stage 1 XGBoost params were found. Falling back to '
        'the committed Stage 1 baseline parameter record.'
    )

best_xgb_params = previous_stage_xgb_params
print('Baseline XGBoost hyperparameters carried forward from Stage 1:')
print(best_xgb_params)

set_seed(SEED)
xgb_model = xgb.XGBClassifier(
    **best_xgb_params,
    random_state=SEED,
    eval_metric='auc',
    n_jobs=N_JOBS,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]


### Baseline Model

We will now evaluate the model against the test set using the following performance indicators:
 - Accuracy
 - Precision
 - Recall
 - AUC

A confusion matrix, ROC and Precision-Recall Curves will also be plotted.

These will be compared against model performance from the model when used with stage 1 data

### Baseline Performance Metrics

In [ ]:
m_id = 5  # unique identifier for this model in the results dictionary

# evaluate the model using the evaluate_and_store_model function defined earlier, 
# which computes performance metrics and stores them in the results dictionary
results = evaluate_and_store_model(
    results,
    m_id=m_id,
    model_name="Baseline",
    model_type="XGBoost",
    stage="Stage 2",
    y_true=y_test,
    y_pred=y_pred_xgb,
    y_prob=y_prob_xgb,
    hyperparameters=best_xgb_params,
    metadata={"threshold": 0.5},
)

# print the evaluation metrics for the XGBoost baseline model in a consistent 
# format using the print_model_metrics function defined earlier
print_model_metrics(results, 2)
print("\nXGBoost Stage 1 (Tuned) Confusion Matrix:")
plot_model_confusion_matrix(results, 2)

print_model_metrics(results, 5)
print("\nXGBoost Stage 2 (Baseline) Confusion Matrix:")
plot_model_confusion_matrix(results, 5)

models_to_plot = build_models_to_plot(results, [2, m_id])
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))



**Comparison - XGBoost Stage 1 Data (Tuned) vs XGBoost Stage 2 Data (Baseline)**

---

AUC increases from 0.9004 to 0.9338, indicating a clear improvement in the model’s ability to separate students who drop out from those who complete across all decision thresholds. This is visible in the ROC curve, where the Stage 2 curve consistently lies above the Stage 1 curve, resulting in a larger area under the curve.


| Metric      | Stage 2 | Stage 1 | Change   | Business Interpretation                                                                                                        |
| ----------- | ------- | ------- | -------- | ------------------------------------------------------------------------------------------------------------------------------ |
| TP          | 480     | 430     | ↑ better | More students who will drop out are correctly identified, improving the ability to target early intervention.                  |
| FN          | 271     | 321     | ↓ better | Fewer at-risk students are missed, reducing the chance of failing to provide support where needed.                             |
| FP          | 142     | 195     | ↓ better | Fewer students who will complete are incorrectly flagged as at risk, reducing unnecessary intervention effort.                 |
| TN          | 4119    | 4066    | ↑ better | More students who will complete are correctly identified, improving overall classification reliability.                        |
| Recall      | 0.639   | 0.573   | ↑ better | A higher proportion of actual dropouts are detected, improving the effectiveness of early warning systems.                     |
| Specificity | 0.967   | 0.954   | ↑ better | Most students who will complete are correctly recognised, meaning intervention resources remain focused on genuine risk cases. |


The precision–recall curve also shows a clear improvement, demonstrating that the Stage 2 model is able to detect more dropout cases while maintaining strong precision. This indicates that the increase in recall is not achieved at the cost of excessive false positives.

When the Stage 1 tuned model is compared with the Stage 2 baseline model, performance improves across all key metrics. This suggests that the additional Stage 2 features provide meaningful predictive information rather than simply increasing model complexity.

The improvement in performance is consistent with the introduction of the absence-related variables, which are likely to be strong indicators of student disengagement. These features allow the model to identify behavioural patterns associated with dropout, leading to better detection of at-risk students without increasing false alarms.

Overall, the results show that the performance gain between Stage 1 and Stage 2 is primarily driven by the addition of new features rather than changes in model tuning, demonstrating the importance of feature engineering in improving predictive performance.

---

### Hyperparameter Tuning

Hyperparameters will now be tuned using the same search space previously used for the Stage 1 model, as summarised below.


| Hyperparameter          |  Search range     |    Baseline | Justification                                                                                          |
| ----------------------- |  ---------------- | ----------: | -------------------------------------------------------------------------------------------------------- |
| learning_rate         |  **0.01 – 0.20**  |        0.10 | Covers slower boosting, through to baseline-fast learning|
| max_depth             |  **3 – 8**        |           3 | Baseline is shallow; range explores moderate complexity    |
| min_child_weight      |  **1 – 12**       |   (default) | Controls minimum leaf weight; higher values regularise and can improve generalisation.                   |
| gamma                 |  **0 – 5**        |   (default) | Penalises splits; encourages simpler trees when >0.                                                      |
| subsample             |  **0.60 – 1.00**  |        0.80 | Row sampling regularisation; includes baseline and stronger/weaker regularisation.                       |
| colsample_bytree      |  **0.60 – 1.00**  |        0.80 | Feature sampling regularisation; includes baseline and explores more/less feature subsampling.           |
| reg_alpha             |  **1e-8 – 10**    | (default=0) | L1 regularisation; small/medium values can help on wide tabular data.                |
| reg_lambda           |  **1e-2 – 50**    | (default=1) | L2 regularisation; explores both weaker and much stronger penalty.                                       |
| n_estimators          |  **3000 (fixed)** |         100 | Not tuned directly; set high and use early stopping to find effective number of trees.                   |
| early_stopping_rounds |  **30 (fixed)**   |           – | Prevents overfitting and replaces manual 'n_estimators' tuning.                                          |


Optuna with Bayesian optimisation will be used again to maintain consistency and comparability with the Stage 1 tuning process.

In [ ]:
set_seed(SEED)

XGB_SEARCH_SPACE = {
    'learning_rate': {'low': 1e-2, 'high': 2e-1, 'log': True},
    'max_depth': {'low': 3, 'high': 8},
    'min_child_weight': {'low': 1.0, 'high': 12.0, 'log': True},
    'gamma': {'low': 0.0, 'high': 5.0},
    'subsample': {'low': 0.60, 'high': 1.00},
    'colsample_bytree': {'low': 0.60, 'high': 1.00},
    'reg_alpha': {'low': 1e-8, 'high': 10.0, 'log': True},
    'reg_lambda': {'low': 1e-2, 'high': 50.0, 'log': True},
}

optuna.logging.set_verbosity(optuna.logging.WARNING)

best_xgb_model, best_xgb_params, best_val_auc, study = run_optuna_xgb(
    n_trials=XGB_N_TRIALS,
    study_name='xgb_stage_2_tabular_auc',
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    seed=SEED,
    n_jobs=N_JOBS,
    overwrite=False,
    artifact_dir=XGB_TUNING_DIR,
    run_search=RUN_XGB_TUNING,
    load_saved=LOAD_SAVED_TUNING,
    resume_search=RESUME_XGB_TUNING,
)

if best_xgb_model is None:
    print('Falling back to the Stage 2 baseline XGBoost model because no stage-specific tuning artefacts are available.')
    best_xgb_model = xgb_model
    best_xgb_params = previous_stage_xgb_params
    best_val_auc = None

print('Best validation AUC:', best_val_auc)
print('Best params:', best_xgb_params)


In [ ]:

trials_path = XGB_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['trials']

if study is not None:
    df_trials = study.trials_dataframe()
elif trials_path.exists():
    df_trials = pd.read_csv(trials_path)
    print(f'Loaded saved XGBoost trial history from {trials_path}')
else:
    df_trials = pd.DataFrame([
        {'number': 0, 'value': best_val_auc, **{f'params_{k}': v for k, v in best_xgb_params.items()}},
    ])
    print('No Stage 2 XGBoost trial history is available. Using the saved best-parameter record only.')

available_hyperparams = [col for col in xgb_hyperparams if col in df_trials.columns]
xgb_results_df_combined = pd.concat(
    [
        xgb_results_df_combined,
        pd.DataFrame(
            {
                'trial_number': df_trials['number'],
                'val_auc': df_trials['value'],
                **{col: df_trials[col] for col in available_hyperparams},
                'Stage': 'Stage 2',
            }
        ),
    ],
    ignore_index=True,
)

display(df_trials.sort_values('value', ascending=False).head(10))


### Tuned Model 

Hyperparameters for the best performing model (trial 855) are shown below and compared to the baseline model:

| Hyperparameter       | Stage 2 Baseline (Stage 1 Tuned) | Stage 2 Tuned  | Interpretation                                                                            |
| -------------------- | -------------------------------- | -------------- | ----------------------------------------------------------------------------------------- |
| **learning_rate**    | 0.200                            | **0.119**      | Lower learning rate in Stage 2 suggests more gradual boosting with the larger feature set |
| **max_depth**        | 7                                | **8**          | Stage 2 allows slightly deeper trees to capture more complex interactions                 |
| **min_child_weight** | 1.21                             | **1.15**       | Both models allow small leaf weights, enabling fine-grained splits                        |
| **gamma**            | 1.02                             | **0.27**       | Lower gamma indicates less split penalty, suggesting clearer signal in Stage 2 features   |
| **subsample**        | 0.880                            | **0.754**      | More row subsampling in Stage 2 provides stronger regularisation                          |
| **colsample_bytree** | 0.681                            | **0.601**      | Increased feature subsampling helps control overfitting with the wider feature set        |
| **reg_alpha (L1)**   | 1.68e-05                         | **0.0052**     | Stage 2 uses slightly stronger L1 regularisation                                          |
| **reg_lambda (L2)**  | 0.0168                           | **4.74**       | Much stronger L2 regularisation, likely due to the increased number of features           |
| **n_estimators**     | Early stopping                   | Early stopping | Both models use early stopping to determine optimal boosting rounds                       |


Because the Stage 1 tuned hyperparameters were reused as the baseline for Stage 2, the differences shown above reflect the effect of retuning the model on the expanded Stage 2 dataset. Overall, the changes in hyperparameters are moderate.

### Tuned Model Evaluation

We will now evaluate the model against the same performance parameters used to evaluate the stage 1 model.


In [ ]:
# make predictions on the test set using the fitted XGBoost model
y_pred_xgb = best_xgb_model.predict(X_test)
y_prob_xgb = best_xgb_model.predict_proba(X_test)[:, 1]

m_id = 6  # unique identifier for this model in the results dictionary
m_ids = [5, 6]  # list of model m_ids to compare in the plots

# store the results in the results dictionary for the tuned XGBoost model
results = evaluate_and_store_model(
    results,
    m_id=m_id,
    model_name="Tuned",
    model_type="XG Boost",
    stage="Stage 2",
    y_true=y_test,
    y_pred=y_pred_xgb,
    y_prob=y_prob_xgb,
    hyperparameters=best_xgb_params,
    metadata={"threshold": 0.5},
)

# print the evaluation metrics and plot the confusion matrix for the tuned XGBoost model

print_model_metrics(results, m_id - 1)
print("\nStage 2 XGBoost Baseline Confusion Matrix:")
plot_model_confusion_matrix(results, m_id - 1)

print_model_metrics(results, m_id)
print("\nStage 2 XGBoost Tuned Confusion Matrix:")
plot_model_confusion_matrix(results, m_id)

models_to_plot = build_models_to_plot(results, m_ids)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))


### Stage 2 vs Stage 1 Comparison

The Stage 2 tuned XGBoost model performs almost identically to the Stage 2 baseline model, with only very small differences across all evaluation metrics.

| Metric      | Baseline   | Tuned      |
| ----------- | ---------- | ---------- |
| Accuracy    | **0.9176** | 0.9168     |
| Precision   | **0.7717** | 0.7694     |
| Recall      | **0.6391** | 0.6352     |
| Specificity | **0.9667** | 0.9664     |
| AUC         | 0.9338     | **0.9340** |


Hyperparameter tuning on the Stage 2 dataset produces only a negligible improvement in performance. Validation AUC during tuning stabilised at approximately 0.93, and the final test AUC (0.9340) is almost identical to the baseline result, indicating that the default configuration was already close to optimal for this dataset.

The ROC and Precision–Recall curves for the baseline and tuned models are nearly indistinguishable, with the tuned model lying only slightly above the baseline across some thresholds. This confirms that tuning does not materially change the model’s classification behaviour.

The confusion matrices also show very similar results. The tuned model produces slightly fewer true positives and slightly more false negatives, while false positives and true negatives remain almost unchanged. These small differences explain the marginal decrease in accuracy, precision, and recall despite the very small increase in AUC.

Overall, these results indicate that the main performance improvement observed in Stage 2 comes from the addition of new features rather than from further hyperparameter tuning. The Stage 1 tuned configuration already provided a strong starting point, and retuning on the expanded dataset yields only minimal gains.

### Feature Importance

We will now review feature importance with the addition of the new features.

Both feature importance and SHAP plots will be created so that we can compare how well they align and use SHAP to better understand magnitude and direction of feature influence.


In [ ]:
# plot the feature importance for the XGBoost model using the feature_importances_ 
# attribute of the fitted model, and display the top 20 most important features
feature_importance = pd.Series(
    best_xgb_model.feature_importances_, index=X_train.columns
).sort_values()
plt.figure(figsize=(10, 35))
feature_importance.plot.barh()
plt.show()

plt.figure(figsize=(10, 10))
feature_importance.iloc[-20:].plot.barh()
plt.show()


In [ ]:
# For the SHAP values, we will use the TreeExplainer which is optimized for 
# tree-based models like XGBoost.
# We will compute the SHAP values for the test set and plot a summary plot to 
# visualize the overall feature importance and the distribution of SHAP values
#  for the top features.

explainer = shap.TreeExplainer(best_xgb_model)
shap_values = explainer.shap_values(X_test)

# plot the SHAP summary plot (beeswarm plot) to show the impact of features on 
# the model's predictions across the test set limit to top 40 features for 
# better visualization
shap.plots.violin(shap_values, X_test, max_display=40, show=False)

fig = plt.gcf()
ax = plt.gca()

# Reduce font size for feature names (y-axis) and SHAP values (x-axis)
ax.tick_params(axis="both", which="major", labelsize=10)

# Reduce font size for labels
ax.xaxis.label.set_size(14)
ax.yaxis.label.set_size(14)

plt.show()


In [ ]:
# Finally, we can also plot the grouped feature importance for both the XGBoost
# feature importance and SHAP values side by side for comparison.
# The features are grouped into categories based on their prefixes

plot_grouped_feature_importance(feature_importance, shap_values, X_val)


**Interpretation**

SHAP importance indicates that absence-related features provide the strongest contribution to model predictions. This suggests that attendance behaviour is the most informative predictor of dropout, and explains the significant improvement in model performance following the addition of these features.



## Stage 2 Neural Network Model

A new Neural Network model will be trained and evaluated on the updated dataset using the best-performing hyperparameters identified during Stage 1. Model performance will then be compared with the Stage 1 results.

A new model is required because the feature set has changed, resulting in different input dimensionality.

In [ ]:

previous_stage_nn_params = load_saved_best_params(PREV_NN_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['best_params'])
if previous_stage_nn_params is None:
    previous_stage_nn_params = {
        'units': 240,
        'layers': 3,
        'activation': 'relu',
        'optimizer': 'adam',
        'lr': 0.0003,
        'dropout': 0.3,
        'l2_strength': 0.0001,
    }
    print(
        'No saved Stage 1 neural-network params were found. Falling '
        'back to the committed Stage 1 baseline parameter record.'
    )

best_nn_params = previous_stage_nn_params
print('Baseline neural-network hyperparameters carried forward from Stage 1:')
print(best_nn_params)

retrain_tuned = False
loaded = False

tf.keras.backend.clear_session()
set_seed(SEED)

model_path = NN_MODEL_DIR / 'baseline_from_stage_1_tuned.keras'
model = build_binary_classifier(input_dim=X_train_s.shape[1], **best_nn_params)

try:
    if not retrain_tuned:
        model, loaded = load_model_weights(model, model_path)
        if loaded:
            print(f'Model loaded from {model_path}')
except Exception as e:
    print(f'Error loading model from {model_path}: {e}')
    print('Proceeding to train the model and save weights for future use.')
    loaded = False

if not loaded:
    early_stop = KerasEarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        min_delta=1e-4,
    )
    history = model.fit(
        X_train_s,
        y_train,
        validation_data=(X_val_s, y_val),
        epochs=50,
        batch_size=64,
        callbacks=[early_stop],
        verbose=1,
    )
    model.save(model_path)


### Baseline Model

We will now evaluate the model against the test set using the following performance indicators:
 - Accuracy
 - Precision
 - Recall
 - AUC

A confusion matrix, ROC and Precision-Recall Curves will also be plotted.

These will be compared against model performance from the model when used with stage 1 data.

### Baseline Performance Metrics

In [ ]:

m_id = 7
m_ids = [4, m_id]

y_prob = model.predict(X_test_s, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

results = evaluate_and_store_model(
    results,
    m_id=m_id,
    model_name='Baseline',
    model_type='Neural Network',
    stage='Stage 2',
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob,
    hyperparameters=best_nn_params,
    metadata={'threshold': 0.5},
)

print_model_metrics(results, m_ids[0])
print()
print('Stage 1 Neural Network Tuned Confusion Matrix:')
plot_model_confusion_matrix(results, m_ids[0])

print_model_metrics(results, m_id)
print()
print('Stage 2 Neural Network Baseline Confusion Matrix:')
plot_model_confusion_matrix(results, m_id)

models_to_plot = build_models_to_plot(results, m_ids)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))


**Comparison of Neural Network Stage 1 (Tuned) vs Neural Network Stage 2 (Baseline)**

---

AUC increases from 0.8899 to 0.9118, indicating improved ability of the model to separate students who drop out from those who complete across all decision thresholds. This is reflected in the ROC curve, where the Stage 2 curve lies consistently above the Stage 1 curve, resulting in a larger area under the curve and improved overall discrimination.

| Metric      | Stage 2 | Stage 1 | Change     | Business Interpretation                                                                                                        |
| ----------- | ------- | ------- | ---------- | ------------------------------------------------------------------------------------------------------------------------------ |
| TP          | 443     | 412     | ↑ better   | More students who will drop out are correctly identified, improving the ability to target early intervention.                  |
| FN          | 308     | 339     | ↓ better   | Fewer at-risk students are missed, reducing the chance of failing to provide support where needed.                             |
| FP          | 156     | 164     | ↓ better   | Fewer students who will complete are incorrectly flagged as at risk, reducing unnecessary intervention effort.                 |
| TN          | 4105    | 4097    | ↑ better   | More students who will complete are correctly identified, improving overall classification reliability.                        |
| Recall      | 0.590   | 0.549   | ↑ better   | A higher proportion of actual dropouts are detected, improving the effectiveness of early warning systems.                     |
| Specificity | 0.963   | 0.962   | ↑ slightly | Most students who will complete are correctly recognised, meaning intervention resources remain focused on genuine risk cases. |


The precision–recall curve also shows an improvement, indicating that the model is able to identify more dropout cases while maintaining strong precision. This suggests that the increase in recall is not achieved at the cost of excessive false positives, which is important in an intervention setting where unnecessary alerts may increase workload.

As with the XGBoost model, retraining the Stage 1 tuned architecture on the Stage 2 dataset leads to improved performance. This indicates that the additional Stage 2 features provide useful predictive signal rather than simply increasing model complexity.

The magnitude of the improvement is slightly smaller than that observed for XGBoost, but the direction of change is consistent, confirming that the performance gain is primarily driven by the additional Stage 2 features rather than changes to the neural network architecture.

---


### Hyperparameter Tuning

Saved tuning artefacts are loaded by default so this notebook can be rerun quickly. Stage 2 neural-network tuning is computationally expensive, so `RUN_NN_TUNING` is set to `False` by default. Set it to `True` only if you want to rerun the RandomSearch workflow and refresh the saved files under `tuning/stage_2/neural_network/`.

Since the Stage 2 dataset introduces additional predictive features but does not fundamentally change the neural-network modelling framework, hyperparameter tuning remains focused on the refined search space identified during Stage 1.


In [ ]:
# we will use a random forest regressor to analyze the importance of the hyperparameters
# in predicting the validation AUC, by fitting a random forest model to the
# hyperparameter values and validation AUC, and then plotting the feature
# importance for each hyperparameter.

hyperparams = ['units', 'layers', 'dropout', 'activation', 'optimizer', 'lr', 'l2_strength']

rf, importances, r2, rmse = analyze_hp_importance(
    results_df_combined,
    hyperparams,
)



SHAP values indicate that learning rate, L2 regularisation strength, and network size (units and layers) have the strongest influence on model performance, with dropout having a smaller but still noticeable effect. Activation function and optimiser choice have comparatively little impact.

The direction and relative importance of these effects are consistent with the earlier tuning stage, so we will continue to use the refined hyperparameter search space.

In [ ]:
tf.keras.backend.clear_session()
set_seed(SEED)

SEARCH_SPACE = {
    'units': {'min_value': 32, 'max_value': 192, 'step': 16},
    'layers': {'min_value': 1, 'max_value': 4},
    'dropout': {'min_value': 0.0, 'max_value': 0.4, 'step': 0.1},
    'activation': ['relu', 'tanh'],
    'optimizer': ['adam'],
    'lr': [3e-4, 1e-3],
    'l2_strength': [0.0, 1e-6, 1e-5, 1e-4],
}

best_hp_values, tuner = run_keras_tuner(
    max_trials=NN_N_TRIALS,
    project_name='nn_tabular_auc_refined_stage_2',
    X_train=X_train_s,
    y_train=y_train,
    X_val=X_val_s,
    y_val=y_val,
    overwrite=False,
    artifact_dir=NN_TUNING_DIR,
    run_search=RUN_NN_TUNING,
    load_saved=LOAD_SAVED_TUNING,
    resume_search=RESUME_NN_TUNING,
)


In [ ]:

nn_trials_path = NN_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['trials']
saved_nn_record = best_params_record(NN_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['best_params'])

if tuner is not None:
    results_df = tuner_trials_to_dataframe(tuner)
elif nn_trials_path.exists():
    results_df = pd.read_csv(nn_trials_path).sort_values('val_auc', ascending=False)
    print(f'Loaded saved Stage 2 NN tuning trials from {nn_trials_path}')
elif saved_nn_record:
    results_df = pd.DataFrame([{
        **saved_nn_record.get('best_params', {}),
        'val_auc': saved_nn_record.get('best_val_auc'),
        'trial_id': 'saved_best_params',
    }])
    print('No Stage 2 NN trial history is available. Using the saved best-parameter record only.')
else:
    results_df = pd.DataFrame()
    print('No Stage 2 NN tuning artefacts are available. Set RUN_NN_TUNING = True to generate them.')

if not results_df.empty:
    display(HTML('<div style="max-height:330px; overflow:auto;">' + results_df.to_html() + '</div>'))
    results_df_combined = pd.concat([results_df_combined, results_df.assign(source='stage2_refined')], ignore_index=True)
else:
    results_df_combined = results_df_combined.copy()


**Refined Hyperparameter Tuning Results**

The best val_auc score from the stage 1 hyperparameter configuration has not been improved. (from 0.9101 to 0.9171)

The strongest models consistently fall within a narrow performance band of approximately 0.914–0.917 validation AUC. 

**Top 5 Neural Network Models – Evaluation**

Small differences in validation AUC may reflect stochastic training variation rather than increased model performance. As per the stage 1 tuning we will attempt to further evaluate model performance and stability as follows:

***Select the top 5 configurations and retrain each:***

- 3 different random seeds + original SEED value to validate repeatability
- batch size = 64
- same early stopping criteria
- same train/validation split

We will compute:

- mean validation AUC
- standard deviation

This may provide a more robust estimate of model performance and identify architectures that are both high performing and stable across training runs.

In [ ]:
top_configs = results_df.sort_values("val_auc", ascending=False).head(5)

print("Top performing hyperparameter combinations from both searches:")
display(
    HTML(
        '<div style="max-height:330px; overflow:auto;">'
        + top_configs.to_html()
        + "</div>"
    )
)


In [ ]:

update_stability = False

filepath = NN_TUNING_DIR / TUNING_ARTIFACT_FILENAMES['stability_data']
if filepath.exists() and not update_stability:
    stability_df = pd.read_hdf(filepath, 'stability')
elif not results_df.empty:
    early_stop = KerasEarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, min_delta=1e-4)
    seeds = STABILITY_SEEDS
    stability_rows = []
    top_configs = results_df.sort_values('val_auc', ascending=False).head(5)

    for cfg_idx, (_, row) in enumerate(top_configs.iterrows(), start=1):
        config = row.to_dict()
        config['units'] = int(config['units'])
        config['layers'] = int(config['layers'])
        config['dropout'] = float(config['dropout'])
        config['lr'] = float(config['lr'])
        config['l2_strength'] = float(config['l2_strength'])

        scores = []
        for seed in seeds:
            tf.keras.backend.clear_session()
            set_seed(seed)
            model = build_binary_classifier(input_dim=X_train_s.shape[1], units=config['units'], layers=config['layers'], activation=config['activation'], optimizer=config['optimizer'], lr=config['lr'], dropout=config['dropout'], l2_strength=config['l2_strength'])
            history = model.fit(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=50, batch_size=64, callbacks=[early_stop], verbose=0)
            scores.append(float(np.max(history.history['val_auc'])))

        stability_rows.append({
            'config_idx': cfg_idx,
            **{k: config[k] for k in ['units', 'layers', 'activation', 'optimizer', 'lr', 'dropout', 'l2_strength']},
            'seed_scores': scores,
            'mean_val_auc': float(np.mean(scores)),
            'std_val_auc': float(np.std(scores)),
        })

    stability_df = pd.DataFrame(stability_rows).sort_values('mean_val_auc', ascending=False)
    stability_df.to_hdf(filepath, key='stability', mode='w')
else:
    stability_df = pd.DataFrame([{
        'config_idx': 1,
        **best_nn_params,
        'seed_scores': [],
        'mean_val_auc': np.nan,
        'std_val_auc': np.nan,
    }])
    print('Skipping Stage 2 NN stability analysis because no saved tuning history is available. Falling back to the saved best parameters.')


In [ ]:
# set to -1 if you want to review the stability results and manually select the 
# best model configuration index for final evaluation on the test set

# Set to the index of the model configuration to use for final evaluation on the 
# test set based on the stability results
model_config_idx = 3  

if MANUAL_REVIEW and model_config_idx == -1:
    raise RuntimeError(
        "Stability optimisation complete.\n"
        "Review results, set SELECTED_CONFIG_IDX in the next cell,\n"
        "then set UPDATE_STABILITY = False and rerun."
    )

elif model_config_idx == -1:
    # default config_idx value from the top performing configuration if not set by user
    model_config_idx = stability_df.iloc[0]["config_idx"]
    print(
        f"No configuration index set for final model selection, defaulting to "
        f"top performing configuration with index {model_config_idx} from the "
        f"stability results."
    )


### Tuned Model

Stability tests for the Stage 2 neural network show the following results:

| Model        | Units | Layers | Activation | Optimizer | LR    | Dropout | L2   | Mean Val AUC | Std    |
|------------|-------|--------|------------|-----------|-------|---------|------|-------------|--------|
| Config_idx 3 | 160 | 3 | relu | adam | 0.001 | 0.3 | 0.0 | **0.9091** | **0.0006** |
| Config_idx 1 | 48  | 1 | tanh | adam | 0.001 | 0.2 | 0.0 | 0.9087 | 0.0059 |
| Config_idx 4 | 128 | 2 | relu | adam | 0.001 | 0.4 | 0.0 | 0.9062 | 0.0016 |
| Config_idx 2 | 176 | 3 | relu | adam | 0.0003 | 0.4 | 1e-4 | 0.9017 | 0.0008 |
| Config_idx 5 | 96  | 4 | relu | adam | 0.0003 | 0.3 | 0.0 | 0.8997 | 0.0009 |

The configuration with 160 units and 3 hidden layers achieved the highest validation AUC while also showing the lowest variance across seeds, indicating stable convergence. Unlike Stage 1, where several configurations performed similarly, the Stage 2 results show a clear best-performing model.

This suggests that the additional Stage 2 features change the optimal network architecture, with a deeper ReLU-based model performing better than the simpler tanh-based networks selected in Stage 1.

Therefore, the final chosen model parameters are:

| Model | Units | Layers | Activation | Optimizer | LR | Dropout | L2 |
|-------|--------|--------|------------|-----------|------|---------|------|
| Final Stage 2 NN | 160 | 3 | relu | adam | 0.001 | 0.3 | 0.0 |


### Tuned Model Evaluation

**Performance Metrics**

We will now evaluate the model against the same performance parameters used to evaluate the stage 1 model.





In [ ]:

tf.keras.backend.clear_session()
set_seed(SEED)

retrain_tuned = False
loaded = False

row = stability_df[stability_df['config_idx'] == model_config_idx].iloc[0]
config = row.to_dict()
config['units'] = int(config['units'])
config['layers'] = int(config['layers'])
config['dropout'] = float(config['dropout'])
config['lr'] = float(config['lr'])
config['l2_strength'] = float(config['l2_strength'])

model_path = NN_MODEL_DIR / f'tuned_model_config_{model_config_idx}.keras'
model = build_binary_classifier(input_dim=X_train_s.shape[1], units=config['units'], layers=config['layers'], activation=config['activation'], optimizer=config['optimizer'], lr=config['lr'], dropout=config['dropout'], l2_strength=config['l2_strength'])

try:
    if not retrain_tuned:
        model, loaded = load_model_weights(model, model_path)
        if loaded:
            print(f'Model loaded from {model_path}')
except Exception as e:
    print(f'Error loading model from {model_path}: {e}')
    print('Proceeding to train the model and save weights for future use.')
    loaded = False

if not loaded:
    early_stop = KerasEarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, min_delta=1e-4)
    history = model.fit(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=50, batch_size=64, callbacks=[early_stop], verbose=1)
    model.save(model_path)


In [ ]:
m_id = 8  # unique identifier for this model in the results dictionary
m_ids = [m_id - 1, m_id]  # list of model m_ids to compare in the plots

# Predict probabilities
y_prob = model.predict(X_test_s, verbose=0).ravel()
# Convert probabilities to binary predictions using a threshold of 0.5
y_pred = (y_prob >= 0.5).astype(int)


results = evaluate_and_store_model(
    results,
    m_id=m_id,
    model_name="Tuned",
    model_type="Neural Network",
    stage="Stage 2",
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob,
    hyperparameters=config,
    metadata={"threshold": 0.5},
)

# print the evaluation metrics for the XGBoost baseline model in a consistent 
# format using the print_model_metrics function defined earlier
print_model_metrics(results, m_ids[0])
print("\nStage 2 Neural Network Baseline Confusion Matrix:")
plot_model_confusion_matrix(results, m_ids[0])


print_model_metrics(results, m_id)
print("\nStage 2 Neural Network Tuned Confusion Matrix:")
plot_model_confusion_matrix(results, m_id)

models_to_plot = build_models_to_plot(
    results,
    m_ids,
)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))


### Stage 2 vs Stage 1 Comparison

The tuned neural network performs better than the baseline model, showing a modest but consistent improvement after hyperparameter tuning.

| Metric      | Baseline   | Tuned      |
| ----------- | ---------- | ---------- |
| Accuracy    | 0.9074     | **0.9094** |
| Precision   | 0.7396     | **0.7583** |
| Recall      | **0.5899** | 0.5806     |
| Specificity | 0.9634     | **0.9674** |
| AUC         | 0.9118     | **0.9220** |


Hyperparameter tuning results in a clear but moderate improvement in overall model performance. Validation AUC during tuning stabilised around 0.91, and the final test AUC of 0.922 indicates that the selected configuration generalises well to unseen data without signs of overfitting.

The ROC curve for the tuned model lies slightly above the baseline curve, showing improved separation between dropout and completion cases. The Precision–Recall curves are very similar, indicating that the improvement mainly affects overall ranking performance rather than substantially changing the precision–recall trade-off.

The confusion matrices show that the tuned model produces fewer false positives and more true negatives, increasing precision and specificity, while recall decreases slightly due to a small increase in false negatives. This indicates that the tuned model makes more conservative predictions, but achieves better overall discrimination between the two classes.

Overall, hyperparameter tuning provides a modest but meaningful improvement over the Stage 2 baseline neural network, although the gain is smaller than the improvement obtained from adding the Stage 2 features themselves.

In [ ]:
## list of tuned models
m_ids = [2, 4, 6, 8]

models_to_plot = build_models_to_plot(
    results,
    m_ids,
)
plot_roc_and_pr_curves(models_to_plot, figsize=(14, 7))


## Stage 2 Model Comparison

### Performance Metrics Comparison (Stage 1 vs Stage 2)

| Model          | Dataset | Accuracy | Precision | Recall | Specificity | AUC    |
| -------------- | ------- | -------- | --------- | ------ | ----------- | ------ |
| XGBoost        | Stage 1 | 0.8970   | 0.6880    | 0.5726 | 0.9542      | 0.9004 |
| XGBoost        | Stage 2 | 0.9168   | 0.7694    | 0.6352 | 0.9664      | 0.9340 |
| Neural Network | Stage 1 | 0.8996   | 0.7153    | 0.5486 | 0.9615      | 0.8899 |
| Neural Network | Stage 2 | 0.9094   | 0.7583    | 0.5806 | 0.9674      | 0.9220 |


**Interpretation**

---

To ensure a fair comparison between Stage 1 and Stage 2, the hyperparameters tuned on the Stage 1 dataset were reused when training both models on the Stage 2 dataset. This allows the effect of the additional features introduced in Stage 2 to be evaluated independently of further hyperparameter optimisation.

Both XGBoost and the neural network show clear performance improvements when trained on the Stage 2 dataset, with higher AUC, recall, and precision compared with Stage 1. This indicates that the additional Stage 2 variables provide useful predictive information. In particular, the absence-related features appear to be strong indicators of lower student engagement, improving the models’ ability to identify dropout cases while maintaining high specificity.

Across both stages, XGBoost achieves the highest overall performance, with the strongest AUC and recall on the Stage 2 dataset. This suggests that the tree-based model is better able to exploit the additional features introduced in Stage 2.

The neural network also benefits from the richer dataset, but the improvement is smaller. This may indicate that the neural network is more sensitive to hyperparameter settings and feature scaling, and therefore gains less from the additional variables when the architecture is not retuned extensively.

Overall, the improvement from Stage 1 to Stage 2 is primarily driven by the addition of new predictive features rather than changes to model structure. The results also show that XGBoost provides slightly better generalisation performance than the neural network for this dataset, making it the strongest candidate for the final model.

---


### Effect of Stage 2 Hyperparameter Tuning

Hyperparameter tuning was repeated on the Stage 2 dataset for both the XGBoost and neural network models to determine whether further optimisation would significantly improve performance compared with the Stage 1 tuned configurations. Performance metrics, AUC values, and ROC / Precision–Recall curves for the tuned models are shown in the respective Stage 2 model sections.

For XGBoost, tuning on the Stage 2 dataset produced only very small changes in performance, with minor differences in AUC, recall, and precision. Validation and test AUC values remained very similar, indicating that the Stage 1 hyperparameters were already close to optimal and that the model generalises well without extensive re-optimisation.

A similar pattern was observed for the neural network. Retuning resulted in modest improvements, particularly in recall and AUC, but the overall change in performance was small. The ROC and Precision–Recall curves for the Stage 1 and Stage 2 tuned models largely overlap, showing that the Stage 1 tuned architecture already provides a good fit for the problem.

Overall, hyperparameter tuning on the Stage 2 dataset does not significantly improve performance for either model. The main performance gain observed in Stage 2 is primarily due to the additional engineered features introduced in the dataset rather than further optimisation of model hyperparameters.

The results suggest that the academic engagement features added in Stage 2 provide useful additional predictive signal, improving the ability of both models to identify students at risk of dropout and supporting more accurate early intervention decisions.